# Installing/Importing Libraries

In [ ]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.0
    Uninstalling sentence-transformers-5.4.0:
      Successfully uninstalled sentence-transformers-5.4.0


In [ ]:
import polars as pl
from sentence_transformers import SentenceTransformer
import torch
import os
import torch


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
foundation_path = f"{drive_path}/processed_data/final_foundation"
output_path = f"{drive_path}/processed_data/variant_llm"
raw_data_path = f"{drive_path}/raw_data"
os.makedirs(output_path, exist_ok=True)

# Loading datasets

In [ ]:
##items_df = pl.read_csv(f"{drive_path}/processed_data/csv_backups/items_metadata_final.csv")
#items_df.write_parquet(f"{foundation_path}/splits/items_metadata_final.parquet")
items_df = pl.read_parquet(f"{foundation_path}/splits/items_metadata_final.parquet")

In [ ]:
items_df.head()

BGGId,Name,Description,YearPublished,GameWeight,AvgRating,BayesAvgRating,StdDev,MinPlayers,MaxPlayers,ComAgeRec,LanguageEase,BestPlayers,GoodPlayers,NumOwned,NumWant,NumWish,NumWeightVotes,MfgPlaytime,ComMinPlaytime,ComMaxPlaytime,MfgAgeRec,NumUserRatings,NumComments,NumAlternates,NumExpansions,NumImplementations,IsReimplementation,Family,Kickstarted,ImagePath,Rank:boardgame,Rank:strategygames,Rank:abstracts,Rank:familygames,Rank:thematic,Rank:cgs,…,Theme_Teaching Programming,Theme_Battle Royal,Theme_Earthquakes,Theme_Bacteria,Theme_Painting / Paintings,Theme_Television (TV) Industry,Theme_Knights Templar,Theme_African Americans,Theme_Hell,Theme_Tiki Cultur,Theme_Astrology,Theme_Video Game Theme: The Oregon Trail,Theme_Māori,Theme_Video Game Theme: Super Mario Bros.,Theme_Mushrooms,Theme_Pulp,Theme_Video Game Theme: Minecraft,Theme_Video Game Theme: Final Fantasy,Theme_Video Game Theme: Carmen Sandiego,Theme_Care Bears,Theme_Hike-Hiking,Theme_Inuit Peoples,Theme_Perfu,Theme_Camping,Theme_Latin American Political Games,Theme_Spanish Political Games,Theme_Video Game Theme: Doo,Theme_Fashion,Theme_Geocaching,Theme_Ecology,Theme_Chernobyl,Theme_Photography,Theme_French Foreign Legion,Theme_Cruise ships,Theme_Apache Tribes,Theme_Rivers,Theme_Flags identification
i64,str,str,i64,f64,f64,f64,f64,i64,i64,f64,f64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,"""Die Macher""","""die macher game seven sequenti…",1986,4.3206,7.61428,7.10363,1.57979,3,5,14.366667,1.395833,5,"""['4', '5']""",7498,501,2039,761,240,240,240,14,5354,0,2,0,0,0,"""Classic Line (Valley Games)""",0,"""https://cf.geekdo-images.com/r…",316,180,21926,21926,21926,21926,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,"""Dragonmaster""","""dragonmaster tricktaking card …",1981,1.963,6.64537,5.78447,1.4544,3,4,12.0,27.0,0,"""[]""",1285,72,191,54,30,30,30,12,562,0,0,0,2,1,null,0,"""https://cf.geekdo-images.com/o…",3993,1577,21926,21926,21926,21926,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,"""Samurai""","""samurai set medieval japan pla…",1998,2.4859,7.45601,7.23994,1.18227,2,4,9.307692,1.0,3,"""['2', '3', '4']""",15578,799,3450,1451,60,30,60,10,15146,0,6,0,1,0,"""Euro Classics (Reiner Knizia)""",0,"""https://cf.geekdo-images.com/o…",224,166,21926,21926,21926,21926,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,"""Tal der Könige""","""triangular box luxurious large…",1992,2.6667,6.60006,5.67954,1.23129,2,4,13.0,256.0,0,"""[]""",638,54,123,30,60,60,60,12,340,0,0,0,0,0,null,0,"""https://cf.geekdo-images.com/n…",5345,21926,21926,21926,21926,21926,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,"""Acquire""","""acquire player strategically i…",1964,2.5031,7.33861,7.14189,1.33583,2,6,11.410256,21.152941,4,"""['3', '4', '5']""",23735,548,2671,1606,90,90,90,12,18655,0,6,2,0,0,"""3M Bookshelf""",0,"""https://cf.geekdo-images.com/3…",290,220,21926,21926,21926,21926,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
items_df = items_df.with_columns(pl.col("Description").fill_null("No description available"))

In [ ]:
raw_mech_cols = pl.read_csv(f"{raw_data_path}/mechanics.csv", n_rows=0).columns
raw_theme_cols = pl.read_csv(f"{raw_data_path}/themes.csv", n_rows=0).columns

In [ ]:
mech_list = [c for c in raw_mech_cols if c != 'BGGId' and c in items_df.columns]
theme_list = [c for c in raw_theme_cols if c != 'BGGId' and c in items_df.columns]

In [ ]:
def extract_tags_from_list(df, column_list, new_name):
    """Concatenates binary flags into a single string based on a provided list of columns."""
    return df.select([
        pl.col("item_id"),
        pl.concat_str(
            [pl.when(pl.col(c) == 1).then(pl.lit(c)).otherwise(pl.lit("")) for c in column_list],
            separator=", "
        ).str.replace_all(", , ", "").str.strip_chars(", ").alias(new_name)
    ])

In [ ]:
mech_tags = extract_tags_from_list(items_df, mech_list, "mech_tags")
theme_tags = extract_tags_from_list(items_df, theme_list, "theme_tags")

# Creating Game Profile

In [ ]:
game_profiles = items_df.select(["item_id", "Name", "Description"]) \
    .join(mech_tags, on="item_id") \
    .join(theme_tags, on="item_id")

In [ ]:
game_profiles = game_profiles.with_columns(
    pl.concat_str([
        pl.lit("Game: "),
        pl.col("Name"),
        pl.lit(". Mechanics: "),
        pl.col("mech_tags").fill_null("None"),
        pl.lit(". Themes: "),
        pl.col("theme_tags").fill_null("None"),
        pl.lit(". Description: "),
        pl.col("Description")
    ]).alias("rich_text_profile")
)

In [ ]:
print(f"Successfully synthesized profiles for {game_profiles.height} games.")
print(f"Sample: {game_profiles['rich_text_profile'][0][:200]}...")

Successfully synthesized profiles for 21925 games.
Sample: Game: Die Macher. Mechanics: Alliances, Area Majority / Influence, Auction/Bidding, Dice Rolling, Hand Management, Simultaneous Action Selection, Negotiation. Themes: Economic, Political. Description:...


# Initialize the high-resolution LLM

In [ ]:
model_name = 'all-mpnet-base-v2'
model = SentenceTransformer(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [ ]:
print(f"Generating 768-dim embeddings using {model_name} on {device}...")
texts = game_profiles["rich_text_profile"].to_list()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

Generating 768-dim embeddings using all-mpnet-base-v2 on cuda...


Batches:   0%|          | 0/343 [00:00<?, ?it/s]

# Creating the final embedding dataframe

In [ ]:
embedding_df = pl.DataFrame({
    "item_id": game_profiles["item_id"],
    "embedding_llm": embeddings.tolist()
})

embedding_df.write_parquet(f"{output_path}/item_embeddings_768.parquet")
print(f"Embeddings saved to {output_path}/item_embeddings_768.parquet")

✅ Success! Embeddings saved to /content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/variant_llm/item_embeddings_768.parquet
